# 🎮 SheepBell - Google Colab 启动笔记本
游戏LQA用 自动视频剪辑与语音转写工具 (Gradio UI)

### 📌 整体流程
1. **运行时设置**: 请确认在菜单栏的 **“代码执行程序” > “更改运行时类型”** 中已选择 **T4 GPU**。
2. **依次运行单元格 1～3**: 打开生成的 `https://xxxx.gradio.live` 链接来处理视频。
3. **运行单元格 4 (后处理)**: 一键在 Google Drive 中自动创建日期文件夹，保存生成物（视频剪辑、图片、CSV）并生成电子表格。

### 1. (推荐) 挂载 Google Drive
为了将成果物（剪辑视频、图片、电子表格）保存到 Google Drive，请首先运行此单元格并允许访问。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. 克隆 / 更新仓库 (首次克隆，之后为拉取更新)

In [ ]:
import os

# SheepBell GitHub 仓库
REPO_URL = "https://github.com/sheep-works/SheepBell"
TARGET_DIR = "/content/SheepBell"

if not os.path.exists(TARGET_DIR):
    print("🚀 正在克隆仓库...")
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print("🔄 正在将现有仓库更新至最新版本 (git pull)...")
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

# 安装依赖项 (包括 GPU 库)
!pip install -q gradio faster-whisper silero-vad onnxruntime-gpu imageio-ffmpeg gspread nvidia-cublas-cu12 nvidia-cudnn-cu12

### 3. 启动 Web UI (生成 Gradio Share URL)
运行后，将在最下方显示 `Running on public URL: https://xxxx.gradio.live`。请点击该链接在浏览器中使用。
*(※ 在使用 UI 期间，请保持此单元格处于运行状态)*

In [ ]:
# 在 GPU 上启动 Gradio UI，并生成公共 URL
!python app.py --share

---
## 🎁 处理完成后的后处理工具
Web UI 中的处理结束后，请运行下方的单元格。

### 4. 🚀 将所有成果物（视频、图片、CSV）保存到 Google Drive ＆ 自动创建电子表格
将 Colab 临时区域的生成物全部复制到 Google Drive 的指定文件夹中，同时自动创建 Google 电子表格并生成链接。

In [ ]:
import os
import csv
import time
import shutil
import datetime
from pathlib import Path
from google.colab import auth
from google.auth import default
import gspread
from googleapiclient.discovery import build

# ==========================================
# 设置: Google Drive 中的目标文件夹路径
# ==========================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/LQA_Results"
SRC_ISSUES_DIR = "/content/SheepBell/issues"

# 1. 在 Google Drive 中创建带时间戳的目标文件夹
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
subfolder_name = f"LQA_{timestamp}"
target_drive_folder = Path(DRIVE_BASE_DIR) / subfolder_name
target_drive_folder.mkdir(parents=True, exist_ok=True)

# 2. 将所有生成的文件 (视频片段、截图、CSV、JSON) 移动到 Google Drive
if os.path.exists(SRC_ISSUES_DIR):
    for item in os.listdir(SRC_ISSUES_DIR):
        s = os.path.join(SRC_ISSUES_DIR, item)
        d = os.path.join(str(target_drive_folder), item)
        if os.path.isfile(s):
            shutil.move(s, d)
    print(f"📁 已将所有文件移动到 Google Drive: {target_drive_folder}")
else:
    print("⚠️ 未找到 'issues' 文件夹。")

# 3. 验证 Google 账户并获取目标文件夹 ID
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

target_folder_id = None
for _ in range(5):
    res = drive_service.files().list(
        q=f"name = '{subfolder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        spaces='drive',
        fields='files(id, name)',
        orderBy='createdTime desc'
    ).execute()
    folders = res.get('files', [])
    if folders:
        target_folder_id = folders[0]['id']
        break
    time.sleep(1)

# 4. 直接在目标文件夹中创建 Google 表格
csv_path = target_drive_folder / "lqa_issues.csv"
if not csv_path.exists():
    csv_path = Path(SRC_ISSUES_DIR) / "lqa_issues.csv"

if csv_path.exists():
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))

    sheet_title = f"LQA_Issues_{timestamp}"
    sh = gc.create(sheet_title, folder_id=target_folder_id)
    worksheet = sh.get_worksheet(0)
    worksheet.update(rows)

    # 双重保障: 如果仍在根目录则将其移动到目标文件夹
    if target_folder_id:
        file_meta = drive_service.files().get(fileId=sh.id, fields='parents').execute()
        current_parents = file_meta.get('parents', [])
        if target_folder_id not in current_parents:
            drive_service.files().update(
                fileId=sh.id,
                addParents=target_folder_id,
                removeParents=','.join(current_parents),
                fields='id, parents'
            ).execute()

    folder_url = f"https://drive.google.com/drive/folders/{target_folder_id}" if target_folder_id else None
    print("\n🎉 已在 Google Drive 中保存所有成果物和表格！")
    if folder_url:
        print(f"📁 打开目标文件夹 -> {folder_url}")
    print(f"🔗 打开表格       -> {sh.url}")
else:
    print("⚠️ 未找到 CSV 文件。")

### 5. 📦 (可选) 批量下载 Zip 文件到本地 PC
如果不使用 Google Drive，而是希望直接将成果物以 Zip 格式下载到本地 PC，请运行此单元格。

In [ ]:
!zip -r /content/issues.zip /content/SheepBell/issues
from google.colab import files
files.download("/content/issues.zip")